# Data cleaning: SA_XenoDataset_2017-2022.csv
**Project:** RAG framework for xenophobic sentiment classification and spatio-temporal health-resource allocation (South Africa)

What this notebook does (mirrors the *Data Cleaning and Preparation* stage of the proposal):

1. Load the raw file safely (tweets contain line breaks, so the file has 25,627 physical lines but only 15,912 records)
2. Standardise the ~80 corrupted `Sentiment` spellings into `Negative / Neutral / Positive`
3. Parse dates and sort chronologically
4. Repair location fields (`Coordinates`, `District`, `Province`, `Tweet Origin`) with spatial imputation
5. Repair numeric fields (`Likes`, `Retweets`, `No of replies`, `Number of Attacks`)
6. Clean tweet text for the LLM/RAG (keeps emojis, case, hashtags, local terms) and for the ML baselines
7. Flag empty / non-linguistic tweets, duplicates and label conflicts
8. Validate and save two files: a full cleaned file and a model-ready file

Nothing is silently dropped: every fix is logged and every imputed value has a flag column.

## 1. Setup and configuration

In [ ]:
!pip install scikit-learn

In [ ]:
import html, re, difflib, unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.neighbors import KNeighborsClassifier

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 90)

# ------------------------- configuration -------------------------
ROOT = Path.cwd()
if ROOT.name == "cleaned":
    ROOT = ROOT.parent
DATA_PATH = ROOT / "SA_XenoDataset_2017-2022.csv"
OUT_DIR = ROOT / "cleaned"
OUT_DIR.mkdir(exist_ok=True)

SENTIMENT_CLASSES = ["Negative", "Neutral", "Positive"]
DISTRICT_CLASSES = ["Urban", "Rural"]
PROVINCES = ["Eastern Cape", "Free State", "Gauteng", "KwaZulu-Natal", "Limpopo",
             "Mpumalanga", "North West", "Northern Cape", "Western Cape"]

SA_LON = (16.0, 33.5)          # rough bounding box of South Africa, used to catch bad coordinates
SA_LAT = (-35.0, -22.0)

# Twitter language codes that mean "no usable language": media-only, mention-only,
# hashtag-only, no linguistic content, undetermined
NON_LINGUISTIC_LANG = {"qme", "qam", "qht", "zxx", "und"}

MIN_LETTERS = 3                # a tweet needs at least this many letters to count as text
DROP_LABEL_CONFLICTS = True    # drop identical tweets that carry different sentiment labels (model-ready file only)
# -----------------------------------------------------------------

log = []
def note(step, n, detail=""):
    log.append({"step": step, "rows_affected": int(n), "detail": detail})
    print(f"[{step}] {int(n):,} rows  {detail}")

## 2. Load the raw data
Read everything as text first (`dtype=str`, `keep_default_na=False`) so pandas cannot silently turn values such as `"NAN"` or `"o"` into something else. Types are set explicitly later.

In [ ]:
raw = pd.read_csv(DATA_PATH, dtype=str, keep_default_na=False)
print("raw shape:", raw.shape)

df = raw.copy()
df.columns = df.columns.str.strip()
df = df.rename(columns={"Unnamed: 0": "raw_id"})      # old index column, kept for traceability

# trim whitespace everywhere, then turn empty strings into real missing values
df = df.apply(lambda s: s.str.strip())
df = df.mask(df.eq(""))

print("\nMissing values per column (after trimming blanks):")
print(df.isna().sum()[lambda s: s > 0])
df.head(3)

## 3. Sentiment labels
`standardise()` normalises each raw value (lower-case, letters only) and maps it to the closest canonical class using fuzzy matching. If no class is a clear best match the label becomes missing rather than being guessed (for example `Nan`, `Nun`, `urban`). The audit table lets you check every change.

In [ ]:
def standardise(series, canonical, cutoff=0.6, margin=0.05):
    """Map messy strings to a canonical list. Exact match after normalising, otherwise the
    closest fuzzy match if it clears `cutoff` and beats the runner-up by `margin`.
    Anything unresolved becomes NaN. Returns (cleaned_series, audit_table)."""
    norm = lambda v: re.sub(r"[^a-z]", "", str(v).lower())
    keys = {norm(c): c for c in canonical}

    def one(v):
        t = norm(v)
        if not t:
            return np.nan
        if t in keys:
            return keys[t]
        scored = sorted(((difflib.SequenceMatcher(None, t, k).ratio(), k) for k in keys), reverse=True)
        if scored[0][0] >= cutoff and (len(scored) == 1 or scored[0][0] - scored[1][0] >= margin):
            return keys[scored[0][1]]
        return np.nan

    mapping = {v: one(v) for v in series.dropna().unique()}
    cleaned = series.map(mapping)
    audit = pd.DataFrame({"raw_value": list(mapping), "mapped_to": list(mapping.values())})
    audit["rows"] = audit["raw_value"].map(series.value_counts())
    audit = audit[audit["raw_value"] != audit["mapped_to"]].sort_values("rows", ascending=False)
    return cleaned, audit


raw_sent = df["Sentiment"].copy()
print("distinct raw Sentiment values:", raw_sent.nunique())

df["Sentiment"], sent_audit = standardise(raw_sent, SENTIMENT_CLASSES)

changed = (raw_sent.fillna("~") != df["Sentiment"].fillna("~")).sum()
note("sentiment standardised", changed, f"{sent_audit['mapped_to'].notna().sum()} variants fixed")
note("sentiment unresolved -> NaN", df["Sentiment"].isna().sum(), "kept as unlabelled, excluded from the model-ready file")

print("\nEvery value that was changed (check this table):")
display(sent_audit.reset_index(drop=True))
print(df["Sentiment"].value_counts(dropna=False))

## 4. Dates and ordering
`DateCreated` and `TimeCreated` are combined into one `datetime`. The raw file is newest-first; it is sorted oldest-first here, which is what the rolling 6-hour and 24-hour sentiment-velocity features need.

> Assumption: timestamps are kept as given. Twitter exports are normally UTC, and South Africa is UTC+2. Convert with `df["datetime"].dt.tz_localize("UTC").dt.tz_convert("Africa/Johannesburg")` if local time matters for your lead-lag analysis.

In [ ]:
df["datetime"] = pd.to_datetime(df["DateCreated"] + " " + df["TimeCreated"], errors="coerce")

bad_dt = df["datetime"].isna()
note("unparseable date/time dropped", bad_dt.sum())
df = df[~bad_dt].copy()

df["raw_id"] = pd.to_numeric(df["raw_id"], errors="coerce").astype("Int64")
df = df.sort_values("datetime", kind="stable").reset_index(drop=True)

print("date range:", df["datetime"].min(), "->", df["datetime"].max())
print(df["datetime"].dt.year.value_counts().sort_index().to_string())

## 5. Location fields
The steps, in order:

* `Province` and `District` are standardised to their valid values; anything else (the file has values like `Neutral` and `Neautral` in these columns) becomes missing.
* `Coordinates` is split into numeric `longitude` / `latitude`. Unparseable values, missing values and points outside South Africa (for example longitude = latitude = -26.3) become missing.
* Missing values are then filled spatially (matching the proposal's "spatial imputation"): coordinates from the tweet's `Tweet Origin`, `Province` from the origin (falling back to a nearest-neighbour vote on coordinates), `District` from the same coordinates or origin.

Each imputed field gets a boolean flag so you can exclude or weight those rows later.

> Note: in this dataset `District` only holds **Urban / Rural**. It is not an administrative district.

In [ ]:
# ---------- helper: fill NaN with the most common value inside progressively coarser groups ----------
def impute_by_mode(frame, target, levels):
    for keys in levels:
        miss = frame[target].isna()
        if not miss.any():
            return
        known = frame.dropna(subset=[target] + keys)
        lookup = (known.groupby(keys + [target]).size().rename("n").reset_index()
                       .sort_values("n", ascending=False).drop_duplicates(keys)
                       .drop(columns="n").rename(columns={target: "_fill"}))
        filled = frame.loc[miss, keys].merge(lookup, on=keys, how="left")["_fill"].to_numpy()
        frame.loc[miss, target] = filled


# ---------- text tidy for place names ----------
df["Tweet Origin"] = df["Tweet Origin"].str.strip(" '\"`").str.replace(r"\s+", " ", regex=True)
df["User Location"] = df["User Location"].str.replace(r"\s+", " ", regex=True)   # free text, left as typed

# ---------- Province / District ----------
df["Province"], prov_audit = standardise(df["Province"], PROVINCES, cutoff=0.8)
df["District"], dist_audit = standardise(df["District"], DISTRICT_CLASSES, cutoff=0.7)
print("Province values changed/nulled:"); display(prov_audit.reset_index(drop=True))
print("District values changed/nulled (top 15):"); display(dist_audit.head(15).reset_index(drop=True))

# ---------- Coordinates -> numeric longitude / latitude ----------
xy = df["Coordinates"].str.extract(r"longitude\s*=\s*(-?\d+\.?\d*)\s*,\s*latitude\s*=\s*(-?\d+\.?\d*)")
df["longitude"] = pd.to_numeric(xy[0], errors="coerce")
df["latitude"] = pd.to_numeric(xy[1], errors="coerce")

in_sa = df["longitude"].between(*SA_LON) & df["latitude"].between(*SA_LAT)
n_invalid = (df["Coordinates"].notna() & ~in_sa).sum()
df.loc[~in_sa, ["longitude", "latitude"]] = np.nan
note("coordinates invalid (unparseable / outside SA) -> NaN", n_invalid)

# ---------- flags, then imputation ----------
df["coord_imputed"] = df["longitude"].isna()
df["province_imputed"] = df["Province"].isna()
df["district_imputed"] = df["District"].isna()

# coordinates: most common valid point for the same Tweet Origin, else the Province median
valid = df.dropna(subset=["longitude", "latitude"])
modal_xy = (valid.groupby(["Tweet Origin", "longitude", "latitude"]).size().rename("n").reset_index()
                 .sort_values("n", ascending=False).drop_duplicates("Tweet Origin")
                 .set_index("Tweet Origin")[["longitude", "latitude"]])
miss = df["longitude"].isna()
for c in ["longitude", "latitude"]:
    df.loc[miss, c] = df.loc[miss, "Tweet Origin"].map(modal_xy[c])
prov_median = valid.groupby("Province")[["longitude", "latitude"]].median()
miss = df["longitude"].isna()
for c in ["longitude", "latitude"]:
    df.loc[miss, c] = df.loc[miss, "Province"].map(prov_median[c])
note("coordinates imputed", df["coord_imputed"].sum())
assert df[["longitude", "latitude"]].notna().all().all(), "some rows still have no coordinates"

# province: from the origin, else nearest neighbours on coordinates
impute_by_mode(df, "Province", [["Tweet Origin"]])
still = df["Province"].isna()
if still.any():
    known = df[~still]
    knn = KNeighborsClassifier(n_neighbors=5).fit(known[["longitude", "latitude"]], known["Province"])
    df.loc[still, "Province"] = knn.predict(df.loc[still, ["longitude", "latitude"]])
note("province imputed", df["province_imputed"].sum())

# district (urban/rural): same coordinates, else same origin, else overall mode.
# This field is noisy inside a place (a single origin has both values), so treat imputed values as weak.
impute_by_mode(df, "District", [["longitude", "latitude"], ["Tweet Origin"]])
df["District"] = df["District"].fillna(df["District"].mode().iat[0])
note("district imputed", df["district_imputed"].sum())

df["Province"].value_counts()

## 6. Numeric fields
* The letter `o` used as a zero (`Number of Attacks`) is corrected to `0`; anything else non-numeric becomes missing.
* Missing engagement counts (`Likes`, `Retweets`, `No of replies`) are set to 0.
* Missing `Number of Attacks` is filled with the value used on the same date and province, then the same date, then 0.
* Outliers are **reported, not removed**. The 20-attack rows on 2017-02-24 belong to a real cluster of events and are the strongest signal in the target.

In [ ]:
COUNT_COLS = ["Likes", "No of replies", "Retweets", "Number of Attacks"]

for c in COUNT_COLS:
    n_typo = df[c].str.fullmatch(r"[oO]").sum()
    if n_typo:
        note(f"{c}: letter 'o' -> 0", n_typo)
    s = df[c].str.replace(r"^[oO]$", "0", regex=True)
    n = pd.to_numeric(s, errors="coerce")
    n = n.where(n >= 0)                       # a negative count is impossible -> NaN
    note(f"{c}: non-numeric -> NaN", n.isna().sum() - df[c].isna().sum())
    df[c] = n

for c in ["Likes", "No of replies", "Retweets"]:
    df[c] = df[c].fillna(0).astype(int)

df["attacks_imputed"] = df["Number of Attacks"].isna()
impute_by_mode(df, "Number of Attacks", [["DateCreated", "Province"], ["DateCreated"]])
df["Number of Attacks"] = df["Number of Attacks"].fillna(0).astype(int)
note("Number of Attacks imputed", df["attacks_imputed"].sum())

print("\nOutlier check (kept as-is):")
display(df[COUNT_COLS].describe().T[["min", "50%", "max"]])
print("Days with the most recorded attacks:")
display(df.groupby("DateCreated")["Number of Attacks"].max().sort_values(ascending=False).head(5))

## 7. Language codes
`Language` is Twitter's own detector, so it is unreliable for South African mixed-language tweets (isiZulu written in English letters is often tagged `en`). It is kept as metadata only. Codes that mean "no language content" are flagged.

In [ ]:
df["Language"] = df["Language"].str.lower()
df["lang_nonlinguistic"] = df["Language"].isin(NON_LINGUISTIC_LANG)
print(df["Language"].value_counts().head(10).to_string())
note("non-linguistic language code flagged", df["lang_nonlinguistic"].sum())

## 8. Tweet text
Two text columns are produced:

| column | use | what it keeps |
|---|---|---|
| `CleanedTweet` | LLM / RAG prompts and embeddings | original case, emojis, hashtags, local-language words, punctuation |
| `text_ml` | TF-IDF + XGBoost baseline | lower-case words, hashtags and apostrophes only |

Both fix HTML entities (`&amp;` -> `&`), strip URLs, replace any `@username` with `@user` (privacy), remove control and invisible characters (emoji joiners are kept) and collapse line breaks and repeated spaces.

In [ ]:
URL_RE = re.compile(r"(https?://\S+|www\.\S+)", re.I)
MENTION_RE = re.compile(r"@\w+")                                  # real handles only; a bare "@ Sandton" (= "at") is kept
INVISIBLE_RE = re.compile(r"[\u0000-\u0008\u000b\u000c\u000e-\u001f\u007f\u200b\u200e\u200f\u2028\u2029\u202a-\u202e\u2060\ufeff]")
LITERAL_ESCAPES_RE = re.compile(r"\\[nrt]")                        # a literal backslash-n typed into the text
WS_RE = re.compile(r"\s+")

def clean_text(x):
    if pd.isna(x):
        return ""
    x = unicodedata.normalize("NFC", str(x))
    for _ in range(2):                                             # handles double-encoded &amp;amp;
        x = html.unescape(x)
    x = URL_RE.sub(" ", x)
    x = MENTION_RE.sub("@user", x)
    x = LITERAL_ESCAPES_RE.sub(" ", x)
    x = INVISIBLE_RE.sub("", x)
    return WS_RE.sub(" ", x).strip()

def ml_text(x):
    x = x.lower().replace("@user", " ")
    x = re.sub(r"[^\w\s'#]", " ", x)                               # drop emojis and punctuation
    x = x.replace("_", " ")
    return WS_RE.sub(" ", x).strip()

before = df["CleanedTweet"].fillna("")
df["CleanedTweet"] = [clean_text(t) for t in df["CleanedTweet"]]
df["text_ml"] = [ml_text(t) for t in df["CleanedTweet"]]
note("tweet text changed by cleaning", (before != df["CleanedTweet"]).sum(), "entities, line breaks, whitespace, etc.")

# usable text = at least MIN_LETTERS letters (rules out empty, emoji-only and "100%"-type tweets)
df["n_letters"] = df["CleanedTweet"].str.count(r"[^\W\d_]")
df["is_valid_text"] = df["n_letters"] >= MIN_LETTERS
note("tweets without usable text", (~df["is_valid_text"]).sum(), "flagged, not deleted")

print("\nBefore / after examples (rows the cleaning changed):")
changed_idx = df.index[(before != df["CleanedTweet"]) & df["is_valid_text"]]
for i in pd.Index(changed_idx).to_series().sample(5, random_state=7):
    print("-", repr(before.loc[i][:110]))
    print("  ->", repr(df.loc[i, "CleanedTweet"][:110]))

## 9. Duplicates and label conflicts
Identical tweets appear several times, sometimes with **different** sentiment labels (copy-paste spam and short replies such as "Are they foreigners?"). For a train/test split this leaks text between sets and adds label noise, so they are flagged here and excluded from the model-ready file. They stay in the full cleaned file because a repeated tweet still counts for time-series and spatial features.

In [ ]:
valid = df["is_valid_text"]

dup = pd.Series(False, index=df.index)
dup[valid] = df.loc[valid, "text_ml"].duplicated(keep="first")      # df is chronological, so the earliest copy is kept
df["is_dup_text"] = dup
note("duplicate tweet text", dup.sum(), "second and later copies")

labelled = valid & df["Sentiment"].notna()
n_labels = df[labelled].groupby("text_ml")["Sentiment"].nunique()
conflicting = set(n_labels[n_labels > 1].index)
df["label_conflict"] = valid & df["text_ml"].isin(conflicting)
note("rows in label-conflict groups", df["label_conflict"].sum(), f"{len(conflicting)} distinct texts")

display(df[df["label_conflict"]].sort_values("text_ml")[["DateCreated", "CleanedTweet", "Sentiment"]].head(8))

## 10. Validate and build the two output tables

In [ ]:
# ---------- assemble ----------
ordered = ["raw_id", "datetime", "DateCreated", "TimeCreated",
           "CleanedTweet", "text_ml", "Sentiment",
           "Likes", "No of replies", "Retweets",
           "Language", "User Location", "Tweet Origin",
           "longitude", "latitude", "District", "Province", "Number of Attacks",
           "coord_imputed", "province_imputed", "district_imputed", "attacks_imputed",
           "lang_nonlinguistic", "n_letters", "is_valid_text", "is_dup_text", "label_conflict"]
df_clean = df[ordered].copy()

keep = df_clean["Sentiment"].notna() & df_clean["is_valid_text"] & ~df_clean["is_dup_text"]
if DROP_LABEL_CONFLICTS:
    keep &= ~df_clean["label_conflict"]
df_ml = df_clean[keep].drop(columns=["n_letters", "is_valid_text", "is_dup_text", "label_conflict"]).reset_index(drop=True)

# ---------- validation ----------
assert df_clean["datetime"].is_monotonic_increasing
assert df_clean["Sentiment"].dropna().isin(SENTIMENT_CLASSES).all()
assert df_clean["Province"].isin(PROVINCES).all()
assert df_clean["District"].isin(DISTRICT_CLASSES).all()
assert df_clean[["longitude", "latitude"]].notna().all().all()
assert df_clean["longitude"].between(*SA_LON).all() and df_clean["latitude"].between(*SA_LAT).all()
assert (df_clean[["Likes", "No of replies", "Retweets", "Number of Attacks"]] >= 0).all().all()
assert not df_clean["CleanedTweet"].str.contains(r"&(?:amp|lt|gt|quot);|[\r\n\t]|\s{2,}").any()
assert df_ml["Sentiment"].notna().all() and (df_ml["CleanedTweet"].str.len() > 0).all()
print("all validation checks passed\n")

# ---------- summary ----------
print("rows: raw", len(raw), "| cleaned", len(df_clean), "| model-ready", len(df_ml))
dist = pd.concat({"df_clean": df_clean["Sentiment"].value_counts(dropna=False),
                  "df_ml": df_ml["Sentiment"].value_counts()}, axis=1).fillna(0).astype(int)
dist.index = dist.index.fillna("Unlabelled")
proposal = pd.Series({"Negative": 9080, "Neutral": 4339, "Positive": 2417}, name="proposal")
display(dist.join(proposal))

print("\nCleaning log")
display(pd.DataFrame(log))

## 11. Save

In [ ]:
df_clean.to_csv(OUT_DIR / "SA_XenoDataset_clean.csv", index=False)        # everything, with flag columns (EDA, GIS, time series)
df_ml.to_csv(OUT_DIR / "SA_XenoDataset_model_ready.csv", index=False)     # labelled, valid, de-duplicated (RAG / baselines)
pd.DataFrame(log).to_csv(OUT_DIR / "cleaning_log.csv", index=False)
print("saved to", OUT_DIR.resolve())
df_ml.head()

**Structuring, Vector Store & Feature Engineering**

In [ ]:
!pip install chromadb sentence-transformers

In [ ]:
import os
import pandas as pd
import chromadb
from chromadb.utils import embedding_functions

# 1. Define Local Paths
DATASET_PATH = "SA_XenoDataset_model_ready.csv"
LOCAL_DB_PATH = "./chroma_db"  # Folder created on your hard drive
COLLECTION_NAME = "xeno_lexicon_context"

# 2. Load Model-Ready Dataset
if not os.path.exists(DATASET_PATH):
    raise FileNotFoundError(f"File '{DATASET_PATH}' not found. Verify your working directory.")

print("Loading dataset...")
df = pd.read_csv(DATASET_PATH)
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

# 3. Initialize Persistent Client (Saves DB directly to LOCAL_DB_PATH)
print(f"Initializing persistent local database at: '{LOCAL_DB_PATH}'")
chroma_client = chromadb.PersistentClient(path=LOCAL_DB_PATH)

# Use standard Sentence Transformer model for embeddings
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create or load collection
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    embedding_function=embedding_fn
)

# 4. Check Existing Records & Populate Local DB
existing_records = collection.count()
print(f"Existing records in local collection: {existing_records}")

if existing_records == 0:
    print("Populating local vector database...")
    
    # Prepare text documents and metadata
    documents = df['CleanedTweet'].fillna('').tolist()
    
    metadata_cols = ['Sentiment', 'Province', 'District', 'DateCreated']
    metadatas = df[metadata_cols].fillna('Unknown').to_dict('records')
    
    ids = [f"tweet_{i}" for i in range(len(df))]
    
    # Insert in batches of 1,000 records
    batch_size = 1000
    for i in range(0, len(df), batch_size):
        end_idx = min(i + batch_size, len(df))
        collection.add(
            documents=documents[i:end_idx],
            metadatas=metadatas[i:end_idx],
            ids=ids[i:end_idx]
        )
        print(f"Saved batch {i} to {end_idx} to local database.")
    
    print(f"Database build complete! Total records saved locally: {collection.count()}")
else:
    print("Local database already exists and contains data. No re-indexing needed.")

In [ ]:
import chromadb
from chromadb.utils import embedding_functions

# Point to the local folder where the database was saved
chroma_client = chromadb.PersistentClient(path="./chroma_db")

embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Connect to the existing collection
collection = chroma_client.get_collection(
    name="xeno_lexicon_context",
    embedding_function=embedding_fn
)

# Query your local vector store
results = collection.query(
    query_texts=["Operation Dudula protest in Gauteng"],
    n_results=3
)

print(results)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import correlate

# 1. Load dataset & convert datetime
df = pd.read_csv("SA_XenoDataset_model_ready.csv")
df['datetime'] = pd.to_datetime(df['datetime'])
df = df.sort_values('datetime').reset_index(drop=True)

# 2. CREATE MISSING COLUMN: Binary indicator for Negative sentiment
df['is_negative'] = (df['Sentiment'] == 'Negative').astype(int)

# 3. Resample daily counts of negative sentiment vs physical attacks
daily = df.set_index('datetime').resample('D').agg({
    'is_negative': 'sum',
    'Number of Attacks': 'max'
}).fillna(0)

# 4. Lead-Lag Temporal Cross-Correlation Calculation
neg_counts = daily['is_negative'] - daily['is_negative'].mean()
attack_counts = daily['Number of Attacks'] - daily['Number of Attacks'].mean()

lags = np.arange(-14, 15)
corr = correlate(neg_counts, attack_counts, mode='full')
corr_lags = corr[len(daily) - 1 - 14 : len(daily) - 1 + 15]

# 5. Plot Lead-Lag Correlation Chart
plt.figure(figsize=(10, 4))
plt.bar(lags, corr_lags, color='teal')
plt.title("Lead-Lag Correlation: Negative Sentiment Surges vs Physical Attacks")
plt.xlabel("Lag (Days) [Negative correlation = Sentiment leads Attacks]")
plt.ylabel("Cross-correlation")
plt.grid(True, linestyle='--', alpha=0.6)
plt.tight_layout()
plt.savefig("lead_lag_correlation.png")
plt.show()